# 07 — Public transport: GTFS import and the transit database

AequilibraE models transit from **GTFS feeds** (the de-facto standard for transit
schedules). Importing a feed creates `public_transport.sqlite` with routes, patterns,
stops, trips and — after map-matching — the real paths through the road network.

The Coquimbo example ships with a GTFS feed for the *Lisanco* operator, which we
import from scratch.


In [1]:
from os import remove
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

from aequilibrae.transit import Transit
from aequilibrae.utils.create_example import create_example

fldr = str(Path(gettempdir()) / uuid4().hex)
project = create_example(fldr, "coquimbo")

# The example ships with a transit DB already built - remove it so we import cleanly
remove(str(Path(fldr) / "public_transport.sqlite"))

In [2]:
data = Transit(project)

gtfs = data.new_gtfs_builder(agency="Lisanco", file_path=str(Path(fldr) / "gtfs_coquimbo.zip"))

# A GTFS feed is a schedule over many days: pick the service day to import.
gtfs.load_date("2016-04-13")

# Map-matching (finding the true road path for each pattern) is optional and slower:
# gtfs.set_allow_map_match(True); gtfs.map_match()

gtfs.save_to_disk()

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

...

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

In [3]:
import geopandas as gpd
import pandas as pd

with project.transit_connection as conn:
    routes = pd.read_sql("SELECT route_id, route, ST_AsText(geometry) wkt FROM routes", conn)
    stops = pd.read_sql("SELECT stop_id, ST_X(geometry) x, ST_Y(geometry) y FROM stops", conn)
    trips = pd.read_sql("SELECT count(*) n FROM trips", conn)

routes_gdf = gpd.GeoDataFrame(routes.drop(columns="wkt"),
                              geometry=gpd.GeoSeries.from_wkt(routes["wkt"]), crs=4326)
stops_gdf = gpd.GeoDataFrame(stops, geometry=gpd.points_from_xy(stops.x, stops.y), crs=4326)

print(f"{len(routes_gdf)} routes, {len(stops_gdf)} stops, {trips.n[0]} trips imported")

2 routes, 78 stops, 360 trips imported


In [4]:
# Maps, cartographic standards and UK geography helpers.
# Model logic stays in the notebook; everything reusable lives in notebooks/uktools/.
from uktools import *


In [5]:
# field()/constant() symbology builders come from the map helper cell

doc = new_map(stops_gdf, zoom=12)
add_gdf(doc, routes_gdf, "routes", symbology=[[constant("#2563eb").encoding("stroke")]])
add_gdf(doc, stops_gdf, "stops", symbology=[[constant("#111827").encoding("fill")]])
doc

[interactive offline map - run the notebook to display]

## Where to go from here

With the transit database in place you can build a **TransitGraph**
(`aequilibrae.transit.TransitGraphBuilder`) and run schedule-based transit
assignment and skimming — see the *public transport assignment* example in the
AequilibraE documentation for the full workflow (hyperpath / optimal-strategies
assignment).


In [6]:
project.close()

---
**Next:** [08 — A full forecasting workflow](08_full_model_workflow.ipynb) ties
notebooks 03-05 together into a base-year/future-year model.
